# Baseline 03: Temporal & Subgroup Comparisons

**Question:** How does the belief network change over time and between groups?

### Part A — Temporal (1975-1985 vs 2010-2020)
1. Build early and late networks
2. Structural comparison (14+ metrics)
3. Differential edges
4. Similarity metrics (GED, spectral)
5. Centrality shift
6. Balance evolution

### Part B — Subgroup (liberal vs conservative, 2000-2010)
7. Conditioned networks
8. Structural comparison
9. Balance comparison
10. Temporal visualization

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from src.loaders.clean_raw_data import clean_datasets
from src.generators.corr_make_network import (
    calculate_correlation_matrix, CorrelationMethod, EdgeSuppressionMethod
)
from src.generators.corr_make_conditioned_network import calculate_conditioned_correlation_matrix
from src.analyzers.matrix_compare import compare_matrices, find_differential_edges
from src.analyzers.graph_similarity import graph_similarity
from src.analyzers.triad_analyzer import count_triads
from src.visualizers.temporal_network_visualizer import generate_temporal_html_visualization

C:\Users\timbo\miniconda3\Lib\site-packages\outdated\utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.0.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
cleaned_df = clean_datasets()

# Common method settings
METHOD = CorrelationMethod.PEARSON
EDGE_SUPP = EdgeSuppressionMethod.REGULARIZATION
SUPP_PARAMS = {'regularization': 0.2}

Loading dataset from from cache...


Done! ✨


---
## Part A: Temporal Comparison (1975-1985 vs 2010-2020)

### 3.1 Build Two-Period Networks

In [3]:
early_years = list(range(1975, 1986))  # 1975-1985
late_years = list(range(2010, 2021, 2))  # 2010, 2012, ..., 2020 (biennial)

print(f"Early period years: {early_years}")
print(f"Late period years: {late_years}")

corr_early = calculate_correlation_matrix(
    cleaned_df, years_of_interest=early_years,
    method=METHOD, partial=True,
    edge_suppression=EDGE_SUPP, suppression_params=SUPP_PARAMS,
    verbose=True
)

corr_late = calculate_correlation_matrix(
    cleaned_df, years_of_interest=late_years,
    method=METHOD, partial=True,
    edge_suppression=EDGE_SUPP, suppression_params=SUPP_PARAMS,
    verbose=True
)

print(f"\nEarly network: {corr_early.shape[0]} variables")
print(f"Late network: {corr_late.shape[0]} variables")

# Report which variables survived in both
common_vars = corr_early.columns.intersection(corr_late.columns)
early_only = set(corr_early.columns) - set(corr_late.columns)
late_only = set(corr_late.columns) - set(corr_early.columns)

print(f"Common variables: {len(common_vars)}")
if early_only:
    print(f"Early-only variables ({len(early_only)}): {sorted(early_only)}")
if late_only:
    print(f"Late-only variables ({len(late_only)}): {sorted(late_only)}")

Early period years: [1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985]
Late period years: [2010, 2012, 2014, 2016, 2018, 2020]

CORRELATION NETWORK STATISTICS
Variables filtered out (metadata): 3
Variables included in analysis: 133
Total number of samples: 13985
--------------------------------------------------


Variables removed due to NaN correlations: 42
Remaining variables after NaN filtering: 91
Removed variables (first 10): WRKWAYUP, NATEDUCY, NATARMSY, NATFAREY, LIBMSLM, RACDIF4, FEFAM, NATCITYY, SPANKING, RACDIF2...
--------------------------------------------------


Sample size statistics for correlations:
  Mean: 30686.0
  Min: 3247
  Max: 71953

CORRELATION NETWORK STATISTICS
Variables filtered out (metadata): 3
Variables included in analysis: 133
Total number of samples: 11771
--------------------------------------------------


Variables removed due to NaN correlations: 12
Remaining variables after NaN filtering: 121
Removed variables (first 10): NATEDUCY, NATARMSY, NATFAREY, NATCITYY, NATRACEY, NATCRIMY, NATSPACY, NATENVIY, NATHEALY, NATAIDY...
--------------------------------------------------


Sample size statistics for correlations:
  Mean: 28338.9
  Min: 3098
  Max: 71953

Early network: 91 variables
Late network: 121 variables
Common variables: 91
Late-only variables (30): ['AFFRMACT', 'COLMSLM', 'FECHLD', 'FEFAM', 'FEPRESCH', 'HELPOTH', 'LIBMSLM', 'NATCHLD', 'NATENRGY', 'NATSCI', 'OBEY', 'POPULAR', 'RACDIF1', 'RACDIF2', 'RACDIF3', 'RACDIF4', 'RELIG_Buddhism', 'RELIG_Christian', 'RELIG_Hinduism', 'RELIG_Inter_nondenominational', 'RELIG_Muslim', 'RELIG_Native_american', 'RELIG_Orthodox_christian', 'RELIG_Other_eastern_religions', 'SPANKING', 'SPKMSLM', 'TEENSEX', 'THNKSELF', 'WORKHARD', 'WRKWAYUP']


### 3.2 Structural Comparison

In [4]:
temporal_comp = compare_matrices(corr_early, corr_late)

print("=== Temporal Comparison: 1975-1985 vs 2010-2020 ===")
for key, val in temporal_comp.items():
    if isinstance(val, dict):
        v1, v2 = val['matrix1'], val['matrix2']
        if isinstance(v1, float):
            print(f"  {key}: {v1:.4f} -> {v2:.4f}")
        else:
            print(f"  {key}: {v1} -> {v2}")
    elif isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

=== Temporal Comparison: 1975-1985 vs 2010-2020 ===
  num_edges: 264 -> 285
  delta_edges: -21
  density: 0.0645 -> 0.0696
  delta_density: -0.0051
  avg_degree: 5.8022 -> 6.2637
  delta_avg_degree: -0.4615
  avg_weight_sum: 0.4077 -> 0.4458
  delta_avg_weight_sum: -0.0382
  clustering_coefficient: 0.4880 -> 0.4370
  delta_clustering_coefficient: 0.0510
  calc_num_triangles: 359 -> 362
  delta_num_triangles: -3
  euclidean_distance: 0.7697
  taxi_cab_distance: 10.5086
  pearson_correlation: 0.9015
  spearman_correlation: 0.6803
  spectral_gap: 0.6896 -> 0.1896
  delta_spectral_gap: 0.5000
  num_communities: 22 -> 23
  delta_num_communities: -1


### 3.3 Differential Edges

In [5]:
stronger_early, stronger_late = find_differential_edges(corr_early, corr_late, top_n=15)

print("=== Top 15 Edges Stronger in EARLY Period (1975-1985) ===")
for var1, var2, diff in stronger_early:
    print(f"  {var1:25s} -- {var2:25s}  diff={diff:+.4f}")

print()
print("=== Top 15 Edges Stronger in LATE Period (2010-2020) ===")
for var1, var2, diff in stronger_late:
    print(f"  {var1:25s} -- {var2:25s}  diff={diff:+.4f}")

=== Top 15 Edges Stronger in EARLY Period (1975-1985) ===


  RELIG_Protestant          -- RELIG_Catholic             diff=+0.2255
  HOMOSEX                   -- XMARSEX                    diff=+0.1174
  NATRACE                   -- HELPBLK                    diff=+0.1171
  PREMARSX                  -- XMARSEX                    diff=+0.0962
  ABDEFECT                  -- ABHLTH                     diff=+0.0797
  SPKATH                    -- SPKRAC                     diff=+0.0793
  SPKCOM                    -- SPKHOMO                    diff=+0.0790
  COLRAC                    -- COLATH                     diff=+0.0784
  CONARMY                   -- CONLEGIS                   diff=+0.0783
  CONBUS                    -- CONFED                     diff=+0.0766
  LIBRAC                    -- LIBHOMO                    diff=+0.0722
  CONARMY                   -- CONFED                     diff=+0.0705
  NATDRUG                   -- NATCRIME                   diff=+0.0657
  LIBRAC                    -- LIBATH                     diff=+0.0630
  COL

### 3.4 Similarity Metrics

In [6]:
# Graph Edit Distance
ged_result = graph_similarity(
    corr_early, corr_late,
    similarity_method='graph_edit_distance',
    edge_threshold=0.0
)
print("Graph Edit Distance:")
print(ged_result)
print()

# Spectral Similarity
try:
    spectral_result = graph_similarity(
        corr_early, corr_late,
        similarity_method='spectral',
        num_eigenvalues=10
    )
    print("Spectral Similarity:")
    print(spectral_result)
except Exception as e:
    print(f"Spectral similarity failed: {e}")
    print("This can happen when the aligned matrices contain NaN/Inf values")
    print("due to variables missing in one period.")

Graph Edit Distance:
SimilarityResult(
  Score: 3180.0000,
  Normalized_score: 0.4380,
  Method: 'graph_edit_distance',
  Metadata: {parameters: {'edge_threshold': 0.0}}
)

Spectral similarity failed: Array must not contain infs or NaNs
This can happen when the aligned matrices contain NaN/Inf values
due to variables missing in one period.


### 3.5 Centrality Shift

In [7]:
def quick_centrality(corr_matrix, top_n=10):
    """Compute centrality using fast networkx built-ins."""
    mat = corr_matrix.copy()
    np.fill_diagonal(mat.values, 0)
    G = nx.from_pandas_adjacency(mat.abs())
    G.remove_edges_from([(u, v) for u, v, d in G.edges(data=True) if d['weight'] == 0])
    
    bc = nx.betweenness_centrality(G, weight='weight')
    strength = {n: sum(d['weight'] for _, _, d in G.edges(n, data=True)) for n in G.nodes()}
    
    bc_df = pd.DataFrame.from_dict(bc, orient='index', columns=['betweenness']).sort_values('betweenness', ascending=False)
    str_df = pd.DataFrame.from_dict(strength, orient='index', columns=['strength']).sort_values('strength', ascending=False)
    return bc_df.head(top_n), str_df.head(top_n)

bc_df_e, str_df_e = quick_centrality(corr_early)
bc_df_l, str_df_l = quick_centrality(corr_late)

print("=== Top 10 by Betweenness: EARLY ===")
print(bc_df_e.to_string())
print()
print("=== Top 10 by Betweenness: LATE ===")
print(bc_df_l.to_string())

=== Top 10 by Betweenness: EARLY ===


          betweenness
NATSPAC      0.401498
LIBATH       0.386517
GRASS        0.297628
EQWLTH       0.251436
NATSOC       0.244444
LIBCOM       0.223970
ABNOMORE     0.204744
CONSCI       0.193258
HELPPOOR     0.192010
SPKHOMO      0.125593

=== Top 10 by Betweenness: LATE ===
                 betweenness
ABANY               0.214846
PRESLAST_DEMREP     0.193277
HOMOSEX             0.161485
CONCLERG            0.139216
FEFAM               0.133193
PORNLAW             0.128992
SPKMSLM             0.126190
GRASS               0.122409
WRKWAYUP            0.115406
POLVIEWS            0.105602


In [8]:
print("=== Top 10 by Strength: EARLY ===")
print(str_df_e.to_string())
print()
print("=== Top 10 by Strength: LATE ===")
print(str_df_l.to_string())

=== Top 10 by Strength: EARLY ===
          strength
LIBCOM    1.087628
COLHOMO   1.047339
ABNOMORE  1.032469
COLATH    1.018067
LIBHOMO   0.980618
LIBATH    0.976811
SPKHOMO   0.949331
PREMARSX  0.936752
ABSINGLE  0.931754
SPKCOM    0.907483

=== Top 10 by Strength: LATE ===
                 strength
PRESLAST_DEMREP  1.961339
HOMOSEX          1.374689
PREMARSX         1.144601
HELPBLK          1.128377
LIBCOM           1.117602
ABSINGLE         1.099139
SPKCOM           1.030427
ABNOMORE         1.010217
SPKMSLM          1.001379
ABANY            0.990096


### 3.6 Balance Evolution

In [9]:
triads_early = count_triads(corr_early, return_names=True)
triads_late = count_triads(corr_late, return_names=True)

def balance_summary(triads, label):
    pos = triads['positive_triads']
    neg = triads['negative_triads']
    total = pos + neg
    print(f"{label}: {pos} balanced / {neg} unbalanced = {100*pos/total:.1f}% balanced (total: {total})")
    if neg > 0:
        print(f"  Unbalanced triads: {triads['negative_triad_nodes']}")

balance_summary(triads_early, 'Early (1975-1985)')
balance_summary(triads_late, 'Late (2010-2020)')

Early (1975-1985): 358 balanced / 1 unbalanced = 99.7% balanced (total: 359)
  Unbalanced triads: [('RELIG_Protestant', 'RELIG_Catholic', 'RELIG_None')]
Late (2010-2020): 566 balanced / 5 unbalanced = 99.1% balanced (total: 571)
  Unbalanced triads: [('OBEY', 'THNKSELF', 'WORKHARD'), ('OBEY', 'THNKSELF', 'HELPOTH'), ('OBEY', 'WORKHARD', 'HELPOTH'), ('THNKSELF', 'WORKHARD', 'HELPOTH'), ('RELIG_Protestant', 'RELIG_Catholic', 'RELIG_None')]


---
## Part B: Subgroup Comparison (Liberal vs Conservative, 2000-2010)

### 3.7 Conditioned Networks

In [10]:
REF_YEARS = list(range(2000, 2011, 2))

# Liberal: POLVIEWS < 0 (left-leaning)
corr_liberal = calculate_conditioned_correlation_matrix(
    cleaned_df, years_of_interest=REF_YEARS,
    method=METHOD, partial=True,
    edge_suppression=EDGE_SUPP, suppression_params=SUPP_PARAMS,
    variable_to_condition='POLVIEWS', condition='less_than_zero',
    verbose=True
)

# Conservative: POLVIEWS > 0 (right-leaning)
corr_conservative = calculate_conditioned_correlation_matrix(
    cleaned_df, years_of_interest=REF_YEARS,
    method=METHOD, partial=True,
    edge_suppression=EDGE_SUPP, suppression_params=SUPP_PARAMS,
    variable_to_condition='POLVIEWS', condition='greater_than_zero',
    verbose=True
)

print(f"Liberal network: {corr_liberal.shape[0]} variables")
print(f"Conservative network: {corr_conservative.shape[0]} variables")


CONDITIONING INFORMATION
Conditioning variable: POLVIEWS
Condition: less_than_zero
Filtered samples count: 17604 of 72390 (24.3%)
--------------------------------------------------

CORRELATION NETWORK STATISTICS
Variables filtered out (metadata): 3
Variables included in analysis: 133
Total number of samples: 3645
--------------------------------------------------
Variables removed due to NaN correlations: 13
Remaining variables after NaN filtering: 120
Removed variables (first 10): NATDRUG, NATHEAL, NATFARE, NATRACE, NATEDUC, NATARMS, NATCRIME, NATENVIR, NATSPAC, NATCITY...
--------------------------------------------------


Sample size statistics for correlations:
  Mean: 7005.4
  Min: 823
  Max: 17604

CONDITIONING INFORMATION
Conditioning variable: POLVIEWS
Condition: greater_than_zero
Filtered samples count: 21122 of 72390 (29.2%)
--------------------------------------------------

CORRELATION NETWORK STATISTICS
Variables filtered out (metadata): 3
Variables included in analysis: 133
Total number of samples: 4636
--------------------------------------------------
Variables removed due to NaN correlations: 14
Remaining variables after NaN filtering: 119
Removed variables (first 10): NATEDUCY, NATARMSY, NATFAREY, NATCITYY, NATRACEY, NATCRIMY, NATSPACY, NATENVIY, NATHEALY, NATAIDY...
--------------------------------------------------


Sample size statistics for correlations:
  Mean: 8748.8
  Min: 817
  Max: 21122
Liberal network: 120 variables
Conservative network: 119 variables


### 3.8 Liberal vs Conservative: Structural Comparison

In [11]:
subgroup_comp = compare_matrices(corr_liberal, corr_conservative)

print("=== Subgroup Comparison: Liberal vs Conservative (2000-2010) ===")
for key, val in subgroup_comp.items():
    if isinstance(val, dict):
        v1, v2 = val['matrix1'], val['matrix2']
        if isinstance(v1, float):
            print(f"  {key}: Liberal={v1:.4f}, Conservative={v2:.4f}")
        else:
            print(f"  {key}: Liberal={v1}, Conservative={v2}")
    elif isinstance(val, float):
        print(f"  {key}: {val:.4f}")
    else:
        print(f"  {key}: {val}")

=== Subgroup Comparison: Liberal vs Conservative (2000-2010) ===
  num_edges: Liberal=368, Conservative=291
  delta_edges: 77
  density: Liberal=0.0649, Conservative=0.0513
  delta_density: 0.0136
  avg_degree: Liberal=6.8785, Conservative=5.4393
  delta_avg_degree: 1.4393
  avg_weight_sum: Liberal=0.4401, Conservative=0.4172
  delta_avg_weight_sum: 0.0229
  clustering_coefficient: Liberal=0.4059, Conservative=0.4211
  delta_clustering_coefficient: -0.0152
  calc_num_triangles: Liberal=507, Conservative=318
  delta_num_triangles: 189
  euclidean_distance: 0.8146
  taxi_cab_distance: 12.5433
  pearson_correlation: 0.9070
  spearman_correlation: 0.6057
  spectral_gap: Liberal=0.0901, Conservative=0.3509
  delta_spectral_gap: -0.2608
  num_communities: Liberal=32, Conservative=30
  delta_num_communities: 2


In [12]:
# Differential edges
stronger_lib, stronger_con = find_differential_edges(corr_liberal, corr_conservative, top_n=15)

print("=== Top 15 Edges Stronger in LIBERAL Network ===")
for var1, var2, diff in stronger_lib:
    print(f"  {var1:25s} -- {var2:25s}  diff={diff:+.4f}")

print()
print("=== Top 15 Edges Stronger in CONSERVATIVE Network ===")
for var1, var2, diff in stronger_con:
    print(f"  {var1:25s} -- {var2:25s}  diff={diff:+.4f}")

=== Top 15 Edges Stronger in LIBERAL Network ===
  HOMOSEX                   -- PRAYER                     diff=+0.1240
  RELIG_Catholic            -- RELIG_None                 diff=+0.1204
  NATENRGY                  -- NATSCI                     diff=+0.1193
  LIBMSLM                   -- TRUST                      diff=+0.1085
  RELIG_Protestant          -- RELIG_None                 diff=+0.1069
  WRKWAYUP                  -- AFFRMACT                   diff=+0.0986
  WRKWAYUP                  -- RACDIF4                    diff=+0.0895
  LIBMSLM                   -- FEFAM                      diff=+0.0867
  RACDIF4                   -- RACDIF2                    diff=+0.0834
  CONCLERG                  -- RELIG_None                 diff=+0.0812
  SPKMSLM                   -- SPKMIL                     diff=+0.0792
  CONCLERG                  -- PREMARSX                   diff=+0.0765
  CONEDUC                   -- CONFED                     diff=+0.0749
  RACDIF4                   

### 3.9 Balance Comparison

In [13]:
triads_lib = count_triads(corr_liberal, return_names=True)
triads_con = count_triads(corr_conservative, return_names=True)

balance_summary(triads_lib, 'Liberal')
balance_summary(triads_con, 'Conservative')

Liberal: 526 balanced / 5 unbalanced = 99.1% balanced (total: 531)
  Unbalanced triads: [('OBEY', 'THNKSELF', 'WORKHARD'), ('OBEY', 'THNKSELF', 'HELPOTH'), ('OBEY', 'WORKHARD', 'HELPOTH'), ('THNKSELF', 'WORKHARD', 'HELPOTH'), ('RELIG_Protestant', 'RELIG_Catholic', 'RELIG_None')]
Conservative: 424 balanced / 5 unbalanced = 98.8% balanced (total: 429)
  Unbalanced triads: [('OBEY', 'THNKSELF', 'WORKHARD'), ('OBEY', 'THNKSELF', 'HELPOTH'), ('OBEY', 'WORKHARD', 'HELPOTH'), ('THNKSELF', 'WORKHARD', 'HELPOTH'), ('RELIG_Protestant', 'RELIG_Catholic', 'RELIG_None')]


### 3.10 Temporal Visualization

Animate the network evolution from 1976 to 2020.

In [14]:
output_path = str(Path.cwd().parent / 'outputs' / 'baseline_temporal_network_1976_2020.html')
Path(output_path).parent.mkdir(parents=True, exist_ok=True)

generate_temporal_html_visualization(
    cleaned_df,
    nodes_to_highlight=['POLVIEWS', 'PARTYID'],
    time_window_length=10,
    start_year=1976,
    end_year=2020,
    step_size=2,
    method=METHOD,
    partial=True,
    edge_suppression=EDGE_SUPP,
    suppression_params=SUPP_PARAMS,
    output_path=output_path
)

print(f"Temporal visualization saved to: {output_path}")

Temporal network visualization has been saved to C:\Users\timbo\Github\BeliefNetworkEvo\outputs\baseline_temporal_network_1976_2020.html
Temporal visualization saved to: C:\Users\timbo\Github\BeliefNetworkEvo\outputs\baseline_temporal_network_1976_2020.html


## Summary

See `analyses/2026-02_baseline-comparisons.md` for the full writeup.